<a href="https://colab.research.google.com/github/lohithshankarappa/bisLAB/blob/main/Week6ParallelCellularAlgo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import random
import math
from concurrent.futures import ThreadPoolExecutor

# ---------------------------------------------------------
# 1 & 2. Environment Representation & Cell Definitions
# ---------------------------------------------------------
FREE = 0
OCCUPIED = 1    # Buildings / physical obstacles
RESTRICTED = 2  # No-fly zones

GRID_SIZE = 10
START = (0, 0)
GOAL = (9, 9)

# Create a 10x10 airspace grid
env_grid = [[FREE for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]

# Add buildings (Occupied = 1)
for r, c in [(2, 2), (2, 3), (3, 2), (3, 3), (6, 5), (6, 6), (7, 5), (7, 6)]:
    env_grid[r][c] = OCCUPIED

# Add restricted areas (Restricted = 2)
for r, c in [(1, 7), (1, 8), (2, 7), (2, 8), (7, 1), (7, 2)]:
    env_grid[r][c] = RESTRICTED


# ---------------------------------------------------------
# 3. Initialize Possible Drone Paths
# ---------------------------------------------------------
def generate_random_path(start, goal, max_steps=35):
    """Generates a random valid-step path from start toward goal."""
    path = [start]
    curr = start
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]

    for _ in range(max_steps):
        if curr == goal:
            break
        candidates = []
        for dr, dc in moves:
            nr, nc = curr[0] + dr, curr[1] + dc
            if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
                candidates.append((nr, nc))
        # Bias slightly toward the goal to encourage convergence
        candidates.sort(key=lambda p: math.dist(p, goal) + random.uniform(0, 2.5))
        curr = candidates[0]
        if curr not in path:
            path.append(curr)
    return path


# ---------------------------------------------------------
# 4. Fitness Evaluation (Distance, Energy, Obstacles)
# ---------------------------------------------------------
def evaluate_fitness(path):
    """Lower score indicates a better (fitter) path."""
    distance = 0.0
    energy = 0.0
    penalty = 0.0

    for i in range(len(path)):
        r, c = path[i]
        # Obstacle and restricted zone penalties
        if env_grid[r][c] == OCCUPIED:
            penalty += 500.0
        elif env_grid[r][c] == RESTRICTED:
            penalty += 1000.0

        if i > 0:
            step_dist = math.dist(path[i - 1], path[i])
            distance += step_dist
            # Diagonal moves consume slightly more energy than cardinal moves
            energy += step_dist * (1.2 if step_dist > 1.0 else 1.0)

        # Turn penalty for sudden direction changes (extra energy drain)
        if i > 1:
            v1 = (path[i - 1][0] - path[i - 2][0], path[i - 1][1] - path[i - 2][1])
            v2 = (path[i][0] - path[i - 1][0], path[i][1] - path[i - 1][1])
            if v1 != v2:
                energy += 0.5

    # Heavy penalty if path fails to reach the goal
    goal_dist = math.dist(path[-1], GOAL)
    if goal_dist > 0:
        penalty += goal_dist * 100.0

    return distance + energy + penalty


# ---------------------------------------------------------
# Evolutionary Operators (Crossover & Mutation)
# ---------------------------------------------------------
def crossover(path1, path2):
    """Single-point crossover at a shared waypoint."""
    common = set(path1[1:-1]).intersection(set(path2[1:-1]))
    if not common:
        return path1[:]
    pivot = random.choice(list(common))
    idx1 = path1.index(pivot)
    idx2 = path2.index(pivot)
    return path1[:idx1] + path2[idx2:]


def mutate(path, mutation_rate=0.3):
    """Perturbs a random waypoint and reconnects."""
    if len(path) < 4 or random.random() > mutation_rate:
        return path
    idx = random.randint(1, len(path) - 2)
    r, c = path[idx]
    dr, dc = random.choice([(-1, 0), (1, 0), (0, -1), (0, 1)])
    nr, nc = r + dr, c + dc
    if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
        new_path = path[:idx] + [(nr, nc)] + path[idx + 1:]
        return new_path
    return path


# ---------------------------------------------------------
# 5. Parallel Cellular Algorithm Execution
# ---------------------------------------------------------
def parallel_cellular_optimization(pop_rows=6, pop_cols=6, generations=60):
    # Initialize 2D cellular grid of candidate solutions
    pop_grid = [
        [generate_random_path(START, GOAL) for _ in range(pop_cols)]
        for _ in range(pop_rows)
    ]

    def update_cell(coord):
        r, c = coord
        # Gather Moore neighborhood (3x3 toroidal wrap-around)
        neighbors = []
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                nr = (r + dr) % pop_rows
                nc = (c + dc) % pop_cols
                neighbors.append(pop_grid[nr][nc])

        # Select best neighbor
        best_neighbor = min(neighbors, key=evaluate_fitness)
        current_path = pop_grid[r][c]

        # Recombine and mutate
        offspring = crossover(current_path, best_neighbor)
        offspring = mutate(offspring)

        # Elitist replacement
        if evaluate_fitness(offspring) < evaluate_fitness(current_path):
            return (r, c, offspring)
        return (r, c, current_path)

    coords = [(r, c) for r in range(pop_rows) for c in range(pop_cols)]

    with ThreadPoolExecutor() as executor:
        for gen in range(generations):
            # Synchronous parallel update across all cellular grid nodes
            results = list(executor.map(update_cell, coords))
            for r, c, updated_path in results:
                pop_grid[r][c] = updated_path

    # Extract globally optimal path from the cellular grid
    all_paths = [pop_grid[r][c] for r in range(pop_rows) for c in range(pop_cols)]
    best_path = min(all_paths, key=evaluate_fitness)
    return best_path, evaluate_fitness(best_path)


if __name__ == "__main__":
    optimal_path, best_cost = parallel_cellular_optimization()
    print(f"Optimized Cost (Fitness): {best_cost:.2f}")
    print(f"Path Length (Steps): {len(optimal_path) - 1}")
    print("Collision-Free Path:", optimal_path)


Optimized Cost (Fitness): 32.78
Path Length (Steps): 13
Collision-Free Path: [(0, 0), (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6), (4, 6), (5, 7), (6, 8), (6, 8), (7, 9), (8, 9), (9, 9)]
